In [1]:
import torch
import pandas as pd
from utils.train import train_model, CustomDistillationDataset, get_balanced_dataloaders
from utils.models import Daowa_maadPrueba
from utils.losses import BinaryDiceLoss, DivergenceKL, BoundaryLoss

modelo = Daowa_maadPrueba(num_clases=1)
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
losses = [
    torch.nn.BCEWithLogitsLoss(),
    BoundaryLoss(), 
    BinaryDiceLoss(smooth=1e-7), 
    DivergenceKL(temperature=4, alpha = 1.0)
]

encoder_params = list(modelo.encoder.parameters())
encoder_ids = set(id(p) for p in encoder_params)
decoder_params = [p for p in modelo.parameters() if id(p) not in encoder_ids]

optimizer = torch.optim.AdamW([
    {'params': encoder_params, 'lr': 1e-5, 'weight_decay': 1e-2},   # Pre-trained → LR bajo
    {'params': decoder_params, 'lr': 1e-4, 'weight_decay': 5e-2},    # Tu decoder → LR normal
])

dataframe = pd.read_csv('labels/dataset_generated.csv')
dataloaders = list(get_balanced_dataloaders(
    csv_labels_path='labels/dataset_generated.csv', 
    imgs_dir='data/oxford/images', 
    human_masks_dir='data/oxford/masks',
    sam_logits_dir='data/oxford/masks_SAM',
    batch_size=16,
    img_size=(384, 384),
    val_split= 0.2,
    test_split= 0.2
))

✅ DataLoaders Balanceados (50/50) Creados Exitosamente.
   - Entrenamiento: 4250 muestras (Gatos: 1402)
   - Validación: 1417 muestras (Gatos: 468)
   - Dev/Test: 1416 muestras (Gatos: 467)


In [2]:
PATIENCE = 8
EPOCHS = 40

train_model(
    modelo = modelo,
    loss_fn = losses,
    optimizador = optimizer,
    dataloaders = dataloaders,
    device_calc = device,
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6),
    epochs=EPOCHS,
    epsilon=1e-8,
    patience=PATIENCE
)

Output()

Iniciando entrenamiento Distilado (2 clases: Fondo, Mascota)


c:\Users\PC\Desktop\Abbadon prueba SAM\source\utils\losses\divergence_kl.py:37: UserWarning: reduction: 'mean' 
divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and 
aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major 
release.
  kl_loss = F.kl_div(student_log_dist, teacher_dist, reduction='mean')

✓ Mejor IoU: 0.9037

✓ Mejor Score: 0.8486 (IoU=0.9037, ValLoss=0.5601)

...

c:\Users\PC\Desktop\Abbadon prueba SAM\source\utils\losses\divergence_kl.py:37: UserWarning: reduction: 'mean' 
divides the total loss by both the batch size and the support size.'batchmean' divides only by the batch size, and 
aligns with the KL div math definition.'mean' will be changed to behave the same as 'batchmean' in the next major 
release.
  kl_loss = F.kl_div(student_log_dist, teacher_dist, reduction='mean')

✓ Mejor Score: 0.9730 (IoU=0.9479, ValLoss=-0.0627)

Epoch 40: Train Loss = -0.0681; Acc = 97.54%; Val Loss = -0.0627, Val Acc = 97.33%, mIoU = 0.9479, Score = 0.9730

LRs: 1.00e-06, 1.00e-06 | IoU: Fondo=0.9495, Mascota=0.9464

'Entrenamiento completado'

In [3]:
# Importa la función desde tu módulo
from utils.metrics.evaluate_model import evaluate_on_unseen_data

# Asegúrate de que el dataloader sea el de testeo/dev
# modelo_entrenado es la variable de tu modelo
# device es tu cuda o cpu
# Se guardarán las métricas y las imágenes en 'source/logs/evaluations'

iou_final, dice_final = evaluate_on_unseen_data(
    model=modelo, 
    dataloader=dataloaders[2], 
    device=device,
    save_dir='logs/evaluations'
)



[Evaluación] Iniciando prueba con datos no vistos...


Evaluando: 100%|██████████| 89/89 [00:18<00:00,  4.87it/s]

--------------------------------------------------
RESULTADOS FINALES EN DATOS NUNCA VISTOS:
👉 IoU (Intersección sobre Unión): 94.59%
👉 F1 / Dice Score:              97.21%
--------------------------------------------------
Visualizaciones guardadas en: logs/evaluations
